# **Exploracion de la oferta de alojamiento de Solola**

Grupo 3 - CC3084 Data Science.

> **Nota sobre 2026:** el inventario corresponde a un corte parcial del primer trimestre. Los registros de 2022-2025 corresponden al cuarto trimestre; por ello, 2026 no debe interpretarse como un cierre anual ni compararse directamente como variacion interanual.


## **1. Preparacion de datos**

### **1.1. Carga de la fuente**

El archivo `oferta_hotelera_2022_2026.csv` contiene el inventario de establecimientos, habitaciones y plazas cama por departamento y tipo de alojamiento, con un corte por ano entre 2022 y 2026. Para este proyecto se seleccionan los registros de Solola, departamento al que pertenecen los municipios del Lago de Atitlan.

Esta fuente describe la capacidad de alojamiento registrada, no la cantidad de visitantes ni las habitaciones que estuvieron ocupadas. La informacion corresponde al departamento y no permite separar la oferta de cada municipio del lago.

In [1]:
from pathlib import Path
import pandas as pd

raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "data" / "raw").is_dir():
    raiz_proyecto = raiz_proyecto.parent

ruta_datos = raiz_proyecto / "data" / "raw" / "oferta_hotelera_2022_2026.csv"
datos = pd.read_csv(ruta_datos, dtype=str, keep_default_na=False, encoding="utf-8")
datos_solola = datos.loc[datos["departamento"].eq("Sololá")].copy()

### **1.2. Primeras cinco filas**

Cada fila corresponde a un tipo de alojamiento de Solola en el corte de un ano. La seleccion contiene 15 observaciones: tres tipos de alojamiento en cinco anos.

In [2]:
datos_solola.head()

,anio,trimestre,departamento,tipo_alojamiento,establecimientos,habitaciones,plazas_cama
54,2022,4,Sololá,Hospedaje,85,830,2188
55,2022,4,Sololá,Hotel,60,989,3037
56,2022,4,Sololá,Pensión,5,58,135
120,2023,4,Sololá,Hospedaje,104,1081,2840
121,2023,4,Sololá,Hotel,54,998,3309


### **1.3. Periodicidad y alcance del inventario**

Aunque existe una columna llamada `trimestre`, la tabla contiene un solo corte por ano, no cuatro observaciones trimestrales por ano. Los registros de 2022 a 2025 indican el cuarto trimestre y los de 2026 el primero. El corte de 2026 no representa el cierre de ese ano y no esta separado del anterior por doce meses.

La [guia de uso de los datos](../data/raw/Guia_de_uso_de_los_datos_UVG.pdf), seccion 5.3, describe esta fuente como un inventario de cortes anuales y aclara que no incluye el alojamiento de plataformas digitales. La guia del proyecto tambien advierte que el levantamiento de 2026 es parcial. No se interpreta esta tabla como una serie trimestral completa ni como el inventario de todo alojamiento existente.
**Aclaracion de comparabilidad:** los registros de 2022-2025 se ubican en el cuarto trimestre, mientras que 2026 corresponde al primero y tiene levantamiento parcial. Son cortes de inventario, no totales de actividad mensual. Por ello, 2026 no se interpretara como un cierre anual comparable directamente con los anos anteriores. El archivo no identifica el dia exacto del corte ni confirma el motivo de esa diferencia o una actualizacion futura.


## **2. Descripcion de las variables**

- **anio:** anio al que corresponde el corte del inventario de alojamiento; variable temporal de tipo cuantitativa discreta.

- **trimestre:** trimestre en el que se ubica el corte del inventario. En esta fuente aparece 4 para 2022-2025 y 1 para 2026; variable temporal cualitativa ordinal. No significa que existan registros de los cuatro trimestres de cada ano.

- **departamento:** departamento al que corresponde el inventario. En la seleccion de este notebook aparece Solola; variable cualitativa nominal.

- **tipo_alojamiento:** categoria del establecimiento segun la fuente: hospedaje, hotel o pension; variable cualitativa nominal. Las categorias no representan una escala de calidad.

- **establecimientos:** cantidad de establecimientos registrados del tipo de alojamiento y corte indicados; variable cuantitativa discreta.

- **habitaciones:** cantidad de habitaciones registradas en esos establecimientos para el corte indicado; variable cuantitativa discreta. Describe capacidad instalada, sin indicar cuantas habitaciones se ocuparon.

- **plazas_cama:** capacidad registrada para alojar personas simultaneamente en esos establecimientos; variable cuantitativa discreta. Una plaza corresponde a capacidad para una persona, por lo que no equivale necesariamente a una cama fisica: una cama doble puede representar dos plazas.

## **3. Revision de calidad de los datos**

### **3.1. Estructura, categorias y faltantes**

Primero se revisa el archivo completo y despues la seleccion de Solola, para distinguir las caracteristicas de la fuente de los casos que afectan al proyecto. Los textos originales se conservan y las conversiones necesarias para el diagnostico se realizan en una copia.

In [3]:
import hashlib
import numpy as np
import unicodedata

huella_inicial = hashlib.sha256(ruta_datos.read_bytes()).hexdigest()
datos_originales = datos.copy(deep=True)
COLUMNAS_ESPERADAS = ["anio","trimestre","departamento","tipo_alojamiento","establecimientos","habitaciones","plazas_cama"]
MARCADORES_AUSENCIA = {"", "null", "none", "nan", "na", "n/a", "s/d"}
TIPOS_ESPERADOS = {"Hospedaje","Hotel","Pensión"}
COLUMNAS_CANTIDAD = ["establecimientos","habitaciones","plazas_cama"]
assert list(datos.columns) == COLUMNAS_ESPERADAS
textos = datos.apply(lambda columna: columna.astype("string").str.strip())
faltantes = textos.apply(lambda columna: columna.isna() | columna.str.casefold().isin(MARCADORES_AUSENCIA))
seleccion_solola = datos["departamento"].eq("Sololá")
display(pd.DataFrame([
    {"alcance":"archivo completo", "filas":len(datos), "departamentos":datos["departamento"].nunique(), "tipos_alojamiento":datos["tipo_alojamiento"].nunique()},
    {"alcance":"Solola", "filas":int(seleccion_solola.sum()), "departamentos":datos_solola["departamento"].nunique(), "tipos_alojamiento":datos_solola["tipo_alojamiento"].nunique()},
]))
display(pd.DataFrame({"faltantes_archivo":faltantes.sum(), "faltantes_solola":faltantes.loc[seleccion_solola].sum(), "espacios_externos":datos.ne(textos).sum()}))
variantes = []
for columna in ["departamento","tipo_alojamiento"]:
    valores = datos[columna].drop_duplicates()
    normalizados = valores.map(lambda valor: unicodedata.normalize("NFC", valor).strip().casefold())
    for _, indices in normalizados.groupby(normalizados).groups.items():
        if len(indices)>1:
            variantes.append({"variable":columna, "variantes":valores.loc[indices].tolist()})
display(pd.DataFrame(variantes, columns=["variable","variantes"]))
display(datos["tipo_alojamiento"].value_counts().to_frame("registros"))
datos.loc[~datos["tipo_alojamiento"].isin(TIPOS_ESPERADOS)]

,alcance,filas,departamentos,tipos_alojamiento
0,archivo completo,330,22,3
1,Solola,15,1,3


,faltantes_archivo,faltantes_solola,espacios_externos
anio,0,0,0
trimestre,0,0,0
departamento,0,0,0
tipo_alojamiento,0,0,0
establecimientos,0,0,0
habitaciones,0,0,0
plazas_cama,0,0,0


,variable,variantes


,registros
tipo_alojamiento,
Hospedaje,110
Hotel,110
Pensión,110


,anio,trimestre,departamento,tipo_alojamiento,establecimientos,habitaciones,plazas_cama


### **3.2. Tipos numericos y coherencia de capacidad**

Anio, trimestre y las tres cantidades deben ser numericos enteros. Las cantidades se revisan como conteos no negativos. Como controles de coherencia se buscan habitaciones menores que establecimientos, plazas menores que habitaciones y establecimientos con capacidad cero. Estas situaciones se senalan para revision; no se corrigen inventando capacidad.

In [4]:
revision = datos.copy(deep=True)
COLUMNAS_NUMERICAS = ["anio","trimestre", *COLUMNAS_CANTIDAD]
for columna in COLUMNAS_NUMERICAS:
    revision[columna] = pd.to_numeric(textos[columna].mask(faltantes[columna]), errors="coerce")
controles = []
for columna in COLUMNAS_NUMERICAS:
    valores = revision[columna]
    controles.append({
        "variable":columna,
        "no_numericos_o_no_finitos":int((~np.isfinite(valores)).sum()),
        "fraccionarios":int((np.isfinite(valores) & valores.mod(1).ne(0)).sum()),
        "negativos":int(valores.lt(0).sum()),
        "ceros_archivo":int(valores.eq(0).sum()),
        "ceros_solola":int(valores.loc[seleccion_solola].eq(0).sum()),
    })
display(pd.DataFrame(controles))
incidencias = pd.DataFrame({
    "anio_fuera_de_2022_2026":~revision["anio"].between(2022,2026),
    "trimestre_fuera_de_1_4":~revision["trimestre"].between(1,4),
    "habitaciones_menores_que_establecimientos":revision["habitaciones"].lt(revision["establecimientos"]),
    "plazas_menores_que_habitaciones":revision["plazas_cama"].lt(revision["habitaciones"]),
    "sin_establecimientos_pero_con_capacidad":revision["establecimientos"].eq(0) & revision[COLUMNAS_CANTIDAD[1:]].gt(0).any(axis=1),
    "con_establecimientos_sin_habitaciones_o_plazas":revision["establecimientos"].gt(0) & revision[COLUMNAS_CANTIDAD[1:]].eq(0).any(axis=1),
})
display(pd.DataFrame({"archivo_completo":incidencias.sum(), "solola":incidencias.loc[seleccion_solola].sum()}))
display(revision.loc[incidencias.any(axis=1)])
revision.loc[seleccion_solola,COLUMNAS_CANTIDAD].agg(["min","max"])

,variable,no_numericos_o_no_finitos,fraccionarios,negativos,ceros_archivo,ceros_solola
0,anio,0,0,0,0,0
1,trimestre,0,0,0,0,0
2,establecimientos,0,0,0,4,0
3,habitaciones,0,0,0,4,0
4,plazas_cama,0,0,0,4,0


,archivo_completo,solola
anio_fuera_de_2022_2026,0,0
trimestre_fuera_de_1_4,0,0
habitaciones_menores_que_establecimientos,0,0
plazas_menores_que_habitaciones,0,0
sin_establecimientos_pero_con_capacidad,0,0
con_establecimientos_sin_habitaciones_o_plazas,0,0


,anio,trimestre,departamento,tipo_alojamiento,establecimientos,habitaciones,plazas_cama


,establecimientos,habitaciones,plazas_cama
min,5,58,135
max,148,1630,5104


### **3.3. Clave, duplicados y cortes disponibles**

La clave de cada observacion es anio, trimestre, departamento y tipo de alojamiento. Se revisa que cada corte tenga las tres categorias y se comparan las combinaciones con los cinco cortes que presenta esta fuente. Los trimestres intermedios que no forman parte del inventario anual no se declaran como datos faltantes a imputar.

In [5]:
clave = ["anio","trimestre","departamento","tipo_alojamiento"]
display(pd.Series({
    "filas_exactamente_repetidas":int(datos.duplicated().sum()),
    "repeticiones_clave_original":int(datos.duplicated(clave).sum()),
    "repeticiones_clave_tipificada":int(revision.duplicated(clave).sum()),
    "repeticiones_clave_solola":int(revision.loc[seleccion_solola].duplicated(clave).sum()),
}).to_frame("cantidad"))
CORTES_ESPERADOS = {(2022,4),(2023,4),(2024,4),(2025,4),(2026,1)}
combinaciones_esperadas = pd.MultiIndex.from_tuples([
    (anio,trimestre,departamento,tipo)
    for anio,trimestre in sorted(CORTES_ESPERADOS)
    for departamento in sorted(datos["departamento"].unique())
    for tipo in sorted(TIPOS_ESPERADOS)
], names=clave)
combinaciones_observadas = pd.MultiIndex.from_frame(revision[clave])
display(combinaciones_esperadas.difference(combinaciones_observadas).to_frame(index=False))
display(combinaciones_observadas.difference(combinaciones_esperadas).to_frame(index=False))
display(revision.groupby(["anio","trimestre"]).agg(registros=("departamento","size"),departamentos=("departamento","nunique"),tipos=("tipo_alojamiento","nunique")))
solola_revision = revision.loc[seleccion_solola].copy()
cortes_solola = solola_revision[["anio","trimestre"]].drop_duplicates().sort_values(["anio","trimestre"])
cortes_solola["separacion_en_trimestres"] = (cortes_solola["anio"]*4+cortes_solola["trimestre"]).diff()
cortes_solola

,cantidad
filas_exactamente_repetidas,0
repeticiones_clave_original,0
repeticiones_clave_tipificada,0
repeticiones_clave_solola,0


,anio,trimestre,departamento,tipo_alojamiento


,anio,trimestre,departamento,tipo_alojamiento


,,registros,departamentos,tipos
anio,trimestre,,,
2022,4,66,22,3
2023,4,66,22,3
2024,4,66,22,3
2025,4,66,22,3
2026,1,66,22,3


,anio,trimestre,separacion_en_trimestres
54,2022,4,<NA>
120,2023,4,4
186,2024,4,4
252,2025,4,4
318,2026,1,1


El archivo tiene 330 filas, equivalentes a 22 departamentos, tres tipos de alojamiento y cinco cortes. La seleccion de Solola tiene 15 filas. No se encontraron duplicados de la clave ni combinaciones ausentes de los cortes esperados.

Los cortes de 2022 a 2025 corresponden al cuarto trimestre y estan separados por cuatro trimestres. El de 2026 corresponde al primero y esta separado del anterior por un trimestre, no por un ano completo. La tabla no permite identificar el dia exacto del levantamiento ni justificar una fecha diaria como si fuera conocida.
La diferencia de momento del corte se mantendra visible al presentar 2026. Los cambios de 2025 a 2026 describen diferencias entre los inventarios disponibles, pero no una variacion interanual entre cierres equivalentes.


### **3.4. Repeticiones entre anos y actualizacion parcial de 2026**

Se contrasta el conjunto completo de establecimientos, habitaciones y plazas cama entre 2025 y 2026, alineando departamento y tipo de alojamiento. La guia del proyecto advierte que 2026 se levanto de forma parcial. La coincidencia de las tres cantidades permite identificar registros sin cambio, pero no demuestra por si sola que se hayan copiado ni que exista un error.

In [6]:
inventario_2025 = revision.loc[revision["anio"].eq(2025)].set_index(["departamento","tipo_alojamiento"])[COLUMNAS_CANTIDAD]
inventario_2026 = revision.loc[revision["anio"].eq(2026)].set_index(["departamento","tipo_alojamiento"])[COLUMNAS_CANTIDAD]
comparacion_cortes = inventario_2025.add_suffix("_2025").join(inventario_2026.add_suffix("_2026"), how="outer", validate="one_to_one")
comparacion_cortes["coinciden_tres_cantidades"] = np.logical_and.reduce([
    comparacion_cortes[f"{columna}_2025"].eq(comparacion_cortes[f"{columna}_2026"])
    for columna in COLUMNAS_CANTIDAD
])
display(pd.Series({
    "combinaciones_comparadas":len(comparacion_cortes),
    "combinaciones_sin_cambio":int(comparacion_cortes["coinciden_tres_cantidades"].sum()),
    "porcentaje_sin_cambio":100*comparacion_cortes["coinciden_tres_cantidades"].mean(),
}).to_frame("valor"))
display(comparacion_cortes.loc["Sololá"])
repetidos_en_distinto_corte = solola_revision.duplicated(["tipo_alojamiento",*COLUMNAS_CANTIDAD], keep=False)
solola_revision.loc[repetidos_en_distinto_corte,COLUMNAS_ESPERADAS]

,valor
combinaciones_comparadas,66.000000
combinaciones_sin_cambio,31.000000
porcentaje_sin_cambio,46.969697


,establecimientos_2025,habitaciones_2025,plazas_cama_2025,establecimientos_2026,habitaciones_2026,plazas_cama_2026,coinciden_tres_cantidades
tipo_alojamiento,,,,,,,
Hospedaje,145,1575,4635,148,1613,4754,False
Hotel,86,1572,4869,90,1630,5104,False
Pensión,5,58,135,6,66,165,False


,anio,trimestre,departamento,tipo_alojamiento,establecimientos,habitaciones,plazas_cama
56,2022,4,Sololá,Pensión,5,58,135
188,2024,4,Sololá,Pensión,5,58,135
254,2025,4,Sololá,Pensión,5,58,135


En el archivo completo, 31 de las 66 combinaciones de departamento y tipo de alojamiento repiten las tres cantidades de 2025, equivalentes al 46.97 %. En Solola, las tres categorias presentan cambios entre 2025 y 2026; esto no demuestra que todo el levantamiento de 2026 sea completo.

En Solola, pension repite 5 establecimientos, 58 habitaciones y 135 plazas cama en 2022, 2024 y 2025. Son cortes de anos diferentes, no filas duplicadas de la misma observacion. Se conservan y se muestra la repeticion como evidencia, sin inferir cierres, aperturas ni arrastre de valores que el archivo no documenta.

### **3.5. Cambios y razones de capacidad para inspeccion**

Se revisan habitaciones por establecimiento y plazas por habitacion para reconocer cambios en la composicion de la oferta. Las razones se calculan sobre el inventario de cada fila y no describen necesariamente un establecimiento individual. No se establece un umbral arbitrario para eliminar registros; tampoco se interpretan los cambios de 2026 como crecimiento interanual comparable.

In [7]:
inspeccion = solola_revision.sort_values(["tipo_alojamiento","anio"]).copy()
inspeccion["habitaciones_por_establecimiento"] = inspeccion["habitaciones"].div(inspeccion["establecimientos"].where(inspeccion["establecimientos"].gt(0)))
inspeccion["plazas_por_habitacion"] = inspeccion["plazas_cama"].div(inspeccion["habitaciones"].where(inspeccion["habitaciones"].gt(0)))
for columna in COLUMNAS_CANTIDAD:
    inspeccion[f"cambio_{columna}"] = inspeccion.groupby("tipo_alojamiento")[columna].diff()
display(inspeccion[["anio","trimestre","tipo_alojamiento",*COLUMNAS_CANTIDAD,"habitaciones_por_establecimiento","plazas_por_habitacion"]])
inspeccion[["anio","trimestre","tipo_alojamiento", *[f"cambio_{columna}" for columna in COLUMNAS_CANTIDAD]]]

,anio,trimestre,tipo_alojamiento,establecimientos,habitaciones,plazas_cama,habitaciones_por_establecimiento,plazas_por_habitacion
54,2022,4,Hospedaje,85,830,2188,9.764706,2.636145
120,2023,4,Hospedaje,104,1081,2840,10.394231,2.627197
186,2024,4,Hospedaje,141,1514,4396,10.737589,2.903567
252,2025,4,Hospedaje,145,1575,4635,10.862069,2.942857
318,2026,1,Hospedaje,148,1613,4754,10.898649,2.947303
55,2022,4,Hotel,60,989,3037,16.483333,3.070779
121,2023,4,Hotel,54,998,3309,18.481481,3.315631
187,2024,4,Hotel,82,1463,4620,17.841463,3.157895
253,2025,4,Hotel,86,1572,4869,18.27907,3.097328
319,2026,1,Hotel,90,1630,5104,18.111111,3.131288


,anio,trimestre,tipo_alojamiento,cambio_establecimientos,cambio_habitaciones,cambio_plazas_cama
54,2022,4,Hospedaje,<NA>,<NA>,<NA>
120,2023,4,Hospedaje,19,251,652
186,2024,4,Hospedaje,37,433,1556
252,2025,4,Hospedaje,4,61,239
318,2026,1,Hospedaje,3,38,119
55,2022,4,Hotel,<NA>,<NA>,<NA>
121,2023,4,Hotel,-6,9,272
187,2024,4,Hotel,28,465,1311
253,2025,4,Hotel,4,109,249
319,2026,1,Hotel,4,58,235


### **3.6. Resultado de la revision y limites de interpretacion**

La estructura revisada no presenta faltantes, espacios externos, tipos de alojamiento ajenos al catalogo, duplicados de clave ni cantidades negativas, fraccionarias o incompatibles con los controles basicos de capacidad. Las 15 observaciones de Solola tienen cantidades positivas; los ceros presentes en otros departamentos se revisan con la coherencia entre establecimientos, habitaciones y plazas, sin trasladarlos a nuestra seleccion.

El tratamiento de 2026 requiere una aclaracion metodologica, no una correccion numerica: es un corte del primer trimestre de un levantamiento parcial y no un cierre anual comparable con 2025. No se reemplazara su trimestre por cuatro para hacer coincidir las etiquetas, ni se rellenaran los trimestres que esta fuente no contiene.

La cobertura es departamental y se limita a las categorias registradas de hospedaje, hotel y pension. No representa la oferta de cada municipio del lago ni incluye automaticamente plataformas digitales. Las cantidades describen inventario disponible, no su uso efectivo ni el total de visitantes. La fuente no identifica individualmente los establecimientos, por lo que no se pueden demostrar aperturas o cierres a partir de las diferencias agregadas.

No se encontraron cantidades que deban corregirse con la evidencia disponible. La seleccion de columnas y el manejo definitivo del corte parcial se discutiran en el bloque de preprocesamiento. Esta seccion deja los hallazgos y sus tablas de respaldo, sin aplicar transformaciones persistentes.

In [8]:
assert datos.equals(datos_originales)
assert hashlib.sha256(ruta_datos.read_bytes()).hexdigest() == huella_inicial
pd.Series({"tabla_original_intacta":True,"archivo_csv_intacto":True,"registros_solola":len(solola_revision)},name="verificacion")

tabla_original_intacta    True
archivo_csv_intacto       True
registros_solola            15
Name: verificacion, dtype: object

## **4. Criterios para la etapa de preprocesamiento**

### **4.1. Seleccion de Solola y columnas de salida**

Primero se cargara el CSV original y se seleccionaran los registros cuyo departamento sea `Sololá`. La seleccion conservara las 15 observaciones disponibles, correspondientes a tres tipos de alojamiento en cinco cortes. El archivo de origen no se modificara.

La tabla procesada sera exclusiva de Solola, por lo que se retirara `departamento` despues de verificar el filtro. Ese alcance territorial se conservara en el nombre del archivo y en su documentacion. La salida tendra seis columnas:

| Columna | Tratamiento previsto |
|---|---|
| `anio` | Se conserva como entero, identificando el ano del corte. |
| `trimestre` | Se conserva como entero entre 1 y 4, para distinguir el momento del corte. |
| `tipo_alojamiento` | Se conserva como categoria: hospedaje, hotel o pension, respetando las etiquetas validas de la fuente. |
| `establecimientos` | Se conserva como cantidad entera no negativa. |
| `habitaciones` | Se conserva como cantidad entera no negativa. |
| `plazas_cama` | Se conserva como cantidad entera no negativa, expresando capacidad de personas. |

Se mantienen anio y trimestre porque juntos identifican el periodo del inventario. No se agregara una fecha diaria como si se conociera el dia de levantamiento ni se reemplazara el trimestre de 2026 por cuatro para uniformar los registros.

### **4.2. Cantidades, repeticiones y ausencia de reparaciones**

La revision de calidad de la seccion 3 ya comprobo que las cantidades son numericas, enteras y positivas en las 15 observaciones de Solola, sin faltantes ni duplicados de clave. Tambien se verifico que las habitaciones no sean menores que los establecimientos y que las plazas cama no sean menores que las habitaciones. No se encontraron valores que deban corregirse con la evidencia disponible.

Se conservaran los inventarios repetidos de pension en 2022, 2024 y 2025, porque corresponden a cortes diferentes. La coincidencia de cantidades no justifica eliminar una observacion, imputarla o inferir que fue copiada. No se aplicaran suavizados, redondeos ni recortes de cantidades.

La clave de la salida sera anio, trimestre y tipo de alojamiento. Si una nueva version repite esa clave, se detendra la consolidacion para revisar las filas, sin conservar una arbitrariamente. Antes de retirar departamento, se validara tambien la clave territorial completa y que todas las filas seleccionadas pertenezcan a Solola.

Si aparecen vacios o marcadores de ausencia en futuras versiones, se representaran como nulos, sin convertirlos en cero ni eliminar automaticamente las filas. Una cantidad no numerica, no finita, negativa o fraccionaria se registrara como incidencia y se rechazara a nulo, manteniendo su valor original en la evidencia. Los ceros no se consideraran errores por si mismos: se contrastaran con las otras cantidades de capacidad. Una inconsistencia de capacidad se registrara para revision, sin inventar el valor que la resolveria.

### **4.3. Tratamiento del corte de 2026**

Se conservaran las tres filas de 2026 y sus valores originales, identificadas como un corte del primer trimestre y con la advertencia de levantamiento parcial. Los inventarios de 2022-2025 pertenecen al cuarto trimestre y son los que se utilizaran para mostrar la evolucion entre cortes del mismo trimestre.

El corte de 2026 podra mostrarse como informacion adicional, separado o rotulado como parcial. Se incluira una nota visible en su presentacion: "2026: inventario parcial del primer trimestre; no comparable directamente con los cortes del cuarto trimestre de 2022-2025". No se presentara como un cierre anual ni se calculara crecimiento interanual comparable con el cierre de 2025. Tampoco se proyectara a fin de ano, se rellenaran trimestres intermedios ni se supondra una actualizacion futura del archivo como parte de la limpieza.

En el archivo completo, 31 de las 66 combinaciones de departamento y tipo repiten las tres cantidades de 2025. En Solola las tres categorias cambian entre 2025 y 2026; esa diferencia no elimina la advertencia de parcialidad ni permite afirmar que el levantamiento del departamento este completo.

### **4.4. Alcance de la capacidad y relaciones con otras fuentes**

Los datos describen la oferta registrada de hospedajes, hoteles y pensiones de Solola, sin desagregacion por municipio. No se distribuiran las habitaciones o plazas entre los pueblos del lago sin una fuente que respalde ese reparto, ni se interpretaran como visitantes o habitaciones efectivamente ocupadas. La cobertura tampoco incluye automaticamente el alojamiento de plataformas digitales.

Las cantidades de las categorias podran agregarse dentro de un mismo corte para describir la capacidad registrada total, conservando el desglose por tipo. No se sumaran inventarios de anos distintos, porque representaria contar capacidad de momentos diferentes como si fuera una sola existencia. El archivo no contiene identificadores individuales de establecimientos y no permite demostrar aperturas o cierres a partir de cambios agregados.

La oferta anual no se utilizara automaticamente como denominador mensual para reconstruir el indice de ocupacion: los cortes, la disponibilidad durante el mes y la cobertura de las fuentes deben ser compatibles. El indice de ocupacion seguira siendo la medida reportada por su propia fuente.

### **4.5. Controles del futuro pipeline**

Los controles que sustentan estas decisiones ya se ejecutaron en la revision de calidad. Cuando se implemente el pipeline se repetiran sobre la salida para comprobar que la seleccion y el tipado no introdujeron errores: seis columnas previstas, tipos correctos, clave unica, cantidades validas y las 15 combinaciones esperadas para esta version.

Tambien se contrastaran los valores de las tres cantidades con la seleccion original, conservando las repeticiones entre anos y la diferenciacion entre cuarto trimestre y primer trimestre. Una futura version con nuevos cortes requerira actualizar el contrato de cobertura de forma explicita; no se descartaran esos datos silenciosamente para forzar las 15 filas actuales.

**Esta seccion documenta el preprocesamiento previsto. Todavia no se han aplicado transformaciones persistentes ni se ha generado un archivo procesado.**
